# 📓 Notebook 02 — Custom Components
**Spendly AI — Coding Camp 2026 powered by DBS Foundation (CC26-PSU276)**

Custom TensorFlow components:
1. CTCLayer — Custom CTC loss layer untuk OCR
2. FocalLoss — Custom loss untuk class imbalance
3. SpendlyCallback — Smart early stopping + logging

In [8]:
import os
import sys
import numpy as np
import tensorflow as tf
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU available: {tf.config.list_physical_devices('GPU')}")

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
SRC_DIR = os.path.join(PROJECT_ROOT, 'src')
os.makedirs(SRC_DIR, exist_ok=True)

TensorFlow version: 2.10.0
GPU available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 2.1 — CTCLayer (Custom Layer)

In [9]:
class CTCLayer(tf.keras.layers.Layer):
    """Custom CTC loss layer untuk OCR model.
    
    Menghitung CTC (Connectionist Temporal Classification) loss
    yang digunakan untuk sequence-to-sequence learning tanpa 
    alignment eksplisit antara input dan output.
    """
    def __init__(self, name="ctc_loss", **kwargs):
        super().__init__(name=name, **kwargs)
        self.loss_fn = tf.keras.backend.ctc_batch_cost

    def call(self, y_true, y_pred):
        batch_len = tf.cast(tf.shape(y_true)[0], dtype="int64")
        input_length = tf.cast(tf.shape(y_pred)[1], dtype="int64")
        label_length = tf.cast(tf.shape(y_true)[1], dtype="int64")
        input_length = input_length * tf.ones(shape=(batch_len, 1), dtype="int64")
        label_length = label_length * tf.ones(shape=(batch_len, 1), dtype="int64")
        loss = self.loss_fn(y_true, y_pred, input_length, label_length)
        self.add_loss(loss)
        return y_pred
    
    def get_config(self):
        config = super().get_config()
        return config

print("CTCLayer defined")

CTCLayer defined


## 2.2 — FocalLoss (Custom Loss)

In [10]:
class FocalLoss(tf.keras.losses.Loss):
    """Focal Loss untuk menangani class imbalance.
    
    Focal Loss menurunkan bobot untuk sampel yang mudah diklasifikasi
    dan fokus pada sampel yang sulit. Cocok untuk dataset yang tidak seimbang.
    
    Args:
        gamma: Focusing parameter. Semakin tinggi, semakin fokus ke hard examples.
        alpha: Weighting factor untuk setiap kelas.
    """
    def __init__(self, gamma=2.0, alpha=0.25, **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma
        self.alpha = alpha

    def call(self, y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        cross_entropy = -y_true * tf.math.log(y_pred)
        weight = self.alpha * y_true * tf.math.pow(1 - y_pred, self.gamma)
        loss = weight * cross_entropy
        return tf.reduce_mean(tf.reduce_sum(loss, axis=-1))
    
    def get_config(self):
        config = super().get_config()
        config.update({"gamma": self.gamma, "alpha": self.alpha})
        return config

print("FocalLoss defined")

FocalLoss defined


## 2.3 — SpendlyCallback (Custom Callback)

In [11]:
class SpendlyCallback(tf.keras.callbacks.Callback):
    """Custom callback: log metrik, early stop cerdas, simpan best model.
    
    Features:
    - Print metrik per epoch dengan format rapi
    - Early stopping berdasarkan val_accuracy dengan min_delta
    - Auto-save best model
    """
    def __init__(self, model_name, patience=10, min_delta=0.001):
        super().__init__()
        self.model_name = model_name
        self.patience = patience
        self.min_delta = min_delta
        self.best_val_acc = 0
        self.wait = 0
        self.best_epoch = 0

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        val_acc = logs.get("val_accuracy", 0)
        train_acc = logs.get("accuracy", 0)
        val_loss = logs.get("val_loss", 0)
        train_loss = logs.get("loss", 0)
        
        print(f"\n[SpendlyCallback] Epoch {epoch+1} | "
              f"train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
              f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f} | "
              f"best={self.best_val_acc:.4f}")
        
        if val_acc > self.best_val_acc + self.min_delta:
            self.best_val_acc = val_acc
            self.best_epoch = epoch + 1
            self.wait = 0
            save_path = os.path.join("models", self.model_name, f"{self.model_name}.keras")
            os.makedirs(os.path.dirname(save_path), exist_ok=True)
            self.model.save(save_path)
            print(f"[SpendlyCallback] Model tersimpan. (best val_acc: {val_acc:.4f})")
        else:
            self.wait += 1
            print(f"[SpendlyCallback] No improvement. Patience: {self.wait}/{self.patience}")
            if self.wait >= self.patience:
                print(f"[SpendlyCallback] Early stop di epoch {epoch+1}. "
                      f"Best epoch: {self.best_epoch} (val_acc: {self.best_val_acc:.4f})")
                self.model.stop_training = True

print("SpendlyCallback defined")

SpendlyCallback defined


## 2.4 — Unit Tests

In [12]:
print("=" * 60)
print("UNIT TESTS")
print("=" * 60)

# Test 1: CTCLayer
print("\n--- Test 1: CTCLayer ---")
try:
    ctc = CTCLayer()
    # Create dummy data
    y_true = tf.constant([[1, 2, 3, 0, 0]], dtype=tf.float32)
    y_pred = tf.random.uniform((1, 10, 80))  # batch=1, timesteps=10, classes=80
    output = ctc(y_true, y_pred)
    assert output.shape == y_pred.shape, f"Shape mismatch: {output.shape} vs {y_pred.shape}"
    print(f"  Output shape: {output.shape}")
    print(f"  Losses: {ctc.losses}")
    print("  PASSED")
except Exception as e:
    print(f"  FAILED: {e}")

# Test 2: FocalLoss
print("\n--- Test 2: FocalLoss ---")
try:
    focal = FocalLoss(gamma=2.0, alpha=0.25)
    y_true = tf.constant([[1, 0, 0], [0, 1, 0], [0, 0, 1]], dtype=tf.float32)
    y_pred = tf.constant([[0.9, 0.05, 0.05], [0.1, 0.8, 0.1], [0.2, 0.3, 0.5]], dtype=tf.float32)
    loss = focal(y_true, y_pred)
    print(f"  Loss value: {loss.numpy():.6f}")
    assert loss.numpy() > 0, "Loss should be positive"
    assert not np.isnan(loss.numpy()), "Loss should not be NaN"
    
    # Edge case: perfect predictions
    y_pred_perfect = tf.constant([[1.0, 0.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 1.0]], dtype=tf.float32)
    loss_perfect = focal(y_true, y_pred_perfect)
    print(f"  Perfect pred loss: {loss_perfect.numpy():.6f}")
    assert loss_perfect.numpy() < loss.numpy(), "Perfect predictions should have lower loss"
    print("  PASSED")
except Exception as e:
    print(f"  FAILED: {e}")

# Test 3: SpendlyCallback
print("\n--- Test 3: SpendlyCallback ---")
try:
    cb = SpendlyCallback(model_name="test_model", patience=3, min_delta=0.001)
    assert cb.patience == 3
    assert cb.min_delta == 0.001
    assert cb.best_val_acc == 0
    assert cb.wait == 0
    print(f"  model_name: {cb.model_name}")
    print(f"  patience: {cb.patience}")
    print(f"  min_delta: {cb.min_delta}")
    print("  PASSED")
except Exception as e:
    print(f"  FAILED: {e}")

# Test config serialization
print("\n--- Test 4: Serialization ---")
try:
    focal_config = focal.get_config()
    print(f"  FocalLoss config: {focal_config}")
    ctc_config = ctc.get_config()
    print(f"  CTCLayer config: {ctc_config}")
    print("  PASSED")
except Exception as e:
    print(f"  FAILED: {e}")

print("\n" + "=" * 60)
print("ALL UNIT TESTS PASSED")
print("=" * 60)

UNIT TESTS

--- Test 1: CTCLayer ---
  Output shape: (1, 10, 80)
  Losses: [<tf.Tensor: shape=(1, 1), dtype=float32, numpy=array([[41.276237]], dtype=float32)>]
  PASSED

--- Test 2: FocalLoss ---
  Loss value: 0.015272
  Perfect pred loss: 0.000000
  PASSED

--- Test 3: SpendlyCallback ---
  model_name: test_model
  patience: 3
  min_delta: 0.001
  PASSED

--- Test 4: Serialization ---
  FocalLoss config: {'reduction': 'auto', 'name': None, 'gamma': 2.0, 'alpha': 0.25}
  CTCLayer config: {'name': 'ctc_loss', 'trainable': True, 'dtype': 'float32'}
  PASSED

ALL UNIT TESTS PASSED


## 2.5 — Export ke src/custom_components.py

In [13]:
component_code = '''"""
Spendly AI — Custom Components
================================
Custom TensorFlow/Keras components untuk Spendly AI project.
- CTCLayer: CTC loss layer untuk OCR model
- FocalLoss: Focal loss untuk class imbalance
- SpendlyCallback: Smart early stopping + model saving

Coding Camp 2026 powered by DBS Foundation (CC26-PSU276)
"""

import os
import tensorflow as tf


class CTCLayer(tf.keras.layers.Layer):
    """Custom CTC loss layer untuk OCR model.
    
    Menghitung CTC (Connectionist Temporal Classification) loss
    yang digunakan untuk sequence-to-sequence learning tanpa 
    alignment eksplisit antara input dan output.
    """
    def __init__(self, name="ctc_loss", **kwargs):
        super().__init__(name=name, **kwargs)
        self.loss_fn = tf.keras.backend.ctc_batch_cost

    def call(self, y_true, y_pred):
        batch_len = tf.cast(tf.shape(y_true)[0], dtype="int64")
        input_length = tf.cast(tf.shape(y_pred)[1], dtype="int64")
        label_length = tf.cast(tf.shape(y_true)[1], dtype="int64")
        input_length = input_length * tf.ones(shape=(batch_len, 1), dtype="int64")
        label_length = label_length * tf.ones(shape=(batch_len, 1), dtype="int64")
        loss = self.loss_fn(y_true, y_pred, input_length, label_length)
        self.add_loss(loss)
        return y_pred
    
    def get_config(self):
        config = super().get_config()
        return config


class FocalLoss(tf.keras.losses.Loss):
    """Focal Loss untuk menangani class imbalance.
    
    Focal Loss menurunkan bobot untuk sampel yang mudah diklasifikasi
    dan fokus pada sampel yang sulit. Cocok untuk dataset yang tidak seimbang.
    
    Args:
        gamma: Focusing parameter. Semakin tinggi, semakin fokus ke hard examples.
        alpha: Weighting factor untuk setiap kelas.
    """
    def __init__(self, gamma=2.0, alpha=0.25, **kwargs):
        super().__init__(**kwargs)
        self.gamma = gamma
        self.alpha = alpha

    def call(self, y_true, y_pred):
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        cross_entropy = -y_true * tf.math.log(y_pred)
        weight = self.alpha * y_true * tf.math.pow(1 - y_pred, self.gamma)
        loss = weight * cross_entropy
        return tf.reduce_mean(tf.reduce_sum(loss, axis=-1))
    
    def get_config(self):
        config = super().get_config()
        config.update({"gamma": self.gamma, "alpha": self.alpha})
        return config


class SpendlyCallback(tf.keras.callbacks.Callback):
    """Custom callback: log metrik, early stop cerdas, simpan best model.
    
    Features:
    - Print metrik per epoch dengan format rapi
    - Early stopping berdasarkan val_accuracy dengan min_delta
    - Auto-save best model
    """
    def __init__(self, model_name, patience=10, min_delta=0.001):
        super().__init__()
        self.model_name = model_name
        self.patience = patience
        self.min_delta = min_delta
        self.best_val_acc = 0
        self.wait = 0
        self.best_epoch = 0

    def on_epoch_end(self, epoch, logs=None):
        logs = logs or {}
        val_acc = logs.get("val_accuracy", 0)
        train_acc = logs.get("accuracy", 0)
        val_loss = logs.get("val_loss", 0)
        train_loss = logs.get("loss", 0)
        
        print(f"\\n[SpendlyCallback] Epoch {epoch+1} | "
              f"train_loss={train_loss:.4f} | train_acc={train_acc:.4f} | "
              f"val_loss={val_loss:.4f} | val_acc={val_acc:.4f} | "
              f"best={self.best_val_acc:.4f}")
        
        if val_acc > self.best_val_acc + self.min_delta:
            self.best_val_acc = val_acc
            self.best_epoch = epoch + 1
            self.wait = 0
            save_path = os.path.join("models", self.model_name, f"{self.model_name}.keras")
            os.makedirs(os.path.dirname(save_path), exist_ok=True)
            self.model.save(save_path)
            print(f"[SpendlyCallback] Model tersimpan. (best val_acc: {val_acc:.4f})")
        else:
            self.wait += 1
            print(f"[SpendlyCallback] No improvement. Patience: {self.wait}/{self.patience}")
            if self.wait >= self.patience:
                print(f"[SpendlyCallback] Early stop di epoch {epoch+1}. "
                      f"Best epoch: {self.best_epoch} (val_acc: {self.best_val_acc:.4f})")
                self.model.stop_training = True
'''

# Write to src/custom_components.py
output_path = os.path.join(SRC_DIR, 'custom_components.py')
with open(output_path, 'w', encoding='utf-8') as f:
    f.write(component_code)

print(f"Exported to: {output_path}")

# Verify import
sys.path.insert(0, SRC_DIR)
import importlib
import custom_components as cc
importlib.reload(cc)
print(f"CTCLayer: {cc.CTCLayer}")
print(f"FocalLoss: {cc.FocalLoss}")
print(f"SpendlyCallback: {cc.SpendlyCallback}")
print("Import verification: PASSED")

Exported to: c:\Nizam\DBS-Foundation\Capstone - Ai Eng - Spendly_ Workspace jilid 3\src\custom_components.py
CTCLayer: <class 'custom_components.CTCLayer'>
FocalLoss: <class 'custom_components.FocalLoss'>
SpendlyCallback: <class 'custom_components.SpendlyCallback'>
Import verification: PASSED


## LAPORAN

In [14]:
print("""
=====================================================
LAPORAN: Notebook 02 -- Custom Components
=====================================================

STATUS: Selesai

HASIL:
- 3 custom components dibuat dan ditest
- CTCLayer: CTC loss layer untuk OCR (tf.keras.layers.Layer)
- FocalLoss: Focal loss gamma=2.0, alpha=0.25 (tf.keras.losses.Loss)
- SpendlyCallback: Early stop + auto-save (tf.keras.callbacks.Callback)
- Semua unit test PASSED

OUTPUT FILES:
- src/custom_components.py (exported)
- notebooks/02_custom_components.py
- notebooks/02_custom_components.ipynb

NEXT: Notebook 03 -- Training Classifier (PRIORITAS UTAMA)
=====================================================
""")


LAPORAN: Notebook 02 -- Custom Components

STATUS: Selesai

HASIL:
- 3 custom components dibuat dan ditest
- CTCLayer: CTC loss layer untuk OCR (tf.keras.layers.Layer)
- FocalLoss: Focal loss gamma=2.0, alpha=0.25 (tf.keras.losses.Loss)
- SpendlyCallback: Early stop + auto-save (tf.keras.callbacks.Callback)
- Semua unit test PASSED

OUTPUT FILES:
- src/custom_components.py (exported)
- notebooks/02_custom_components.py
- notebooks/02_custom_components.ipynb

NEXT: Notebook 03 -- Training Classifier (PRIORITAS UTAMA)

